In [1]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage,HumanMessage
from langchain.agents import AgentState,create_agent
from langgraph.checkpoint.memory import InMemorySaver


from dotenv import load_dotenv
import os

load_dotenv()
MODEL=os.getenv("MODEL","gpt-4o-mini")
print(f"{MODEL=}")

MODEL='gpt-4o-mini'


In [2]:
class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [ ]:
agent = create_agent(
    MODEL,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)
agent

In [11]:
config={"configurable": {"thread_id": "1"}}
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    config=config
)

In [12]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='dd93b7f7-18d0-4487-b472-04aa57b695f0'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 60, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8c9579f592', 'id': 'chatcmpl-EUWgUO5OcYaJWqO5OhyHmMptbzIEj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0fc97-8fea-7890-80f6-49d1877b6342-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'

In [15]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="What is my fav color")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)
pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='What is my fav color', additional_kwargs={}, response_metadata={}, id='c246e85f-1ce6-4dc7-8812-cd205c2a8dd4'),
              AIMessage(content="I don't know your favourite colour yet. Would you like to share it with me?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 60, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8c9579f592', 'id': 'chatcmpl-EUWgmcmk9njia63wI3DD5V4KnGU6M', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0fc97-d5a1-7a51-9387-be03a0a7f0e3-0

## Read state

In [16]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    MODEL,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [17]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='06e7ffbd-c8ab-42a8-a625-e6f3895569bd'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 81, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_169c0f824d', 'id': 'chatcmpl-EUWhZtnYuMGzyQK7nhCIerKVC0zvo', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0fc98-9504-7dd2-9dcf-13b7d64371bd-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'

In [18]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='06e7ffbd-c8ab-42a8-a625-e6f3895569bd'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 81, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_169c0f824d', 'id': 'chatcmpl-EUWhZtnYuMGzyQK7nhCIerKVC0zvo', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0fc98-9504-7dd2-9dcf-13b7d64371bd-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'